# Lesson 12 — Interior-Point Methods

## Learning objectives

1. State the path-following IPM for LP.
2. Walk through one Mehrotra predictor-corrector step {cite:p}`Mehrotra1992`.
3. Compare IPM vs simplex on tall, fat, and dense LPs.
4. Recognize crossover (turning IPM solution into a basic solution).
5. Read an IPM's trajectory ($\mu$, primal and dual infeasibility, step lengths) from POUNCE's solve report.

## 1. The KKT system for LP

For $\min c^\top x$ s.t. $Ax = b, x \ge 0$, the KKT conditions are

$$Ax = b, \quad A^\top y + s = c, \quad x_i s_i = 0, \quad x, s \ge 0.$$

Replace the complementarity $x_i s_i = 0$ with $x_i s_i = \mu$ for a *barrier parameter* $\mu > 0$. Solve this perturbed system, then drive $\mu \to 0$. The locus of solutions is the **central path** {cite:p}`Wright1997`.

## 2. Newton step on the perturbed KKT

$$\begin{bmatrix} 0 & A^\top & I \\ A & 0 & 0 \\ S & 0 & X \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \\ \Delta s \end{bmatrix} = \begin{bmatrix} c - A^\top y - s \\ b - Ax \\ \mu e - X S e \end{bmatrix}$$

with $X = \text{diag}(x), S = \text{diag}(s), e = (1, ..., 1)^\top$. Eliminate $\Delta s$ to get a smaller normal-equation system $A D A^\top \Delta y = r$ with $D = X S^{-1}$ — solve via Cholesky.

## 3. Mehrotra's predictor-corrector

A practical IPM {cite:p}`Mehrotra1992`:

1. **Predictor:** solve the linear system with $\mu = 0$ on the right-hand side (the affine-scaling direction $\Delta^{\text{aff}}$). Compute the maximal step $\alpha^{\text{aff}} \in (0, 1]$ keeping $x, s$ positive, and the post-step duality measure $\mu_{\text{aff}} = (x + \alpha^{\text{aff}} \Delta x^{\text{aff}})^\top (s + \alpha^{\text{aff}} \Delta s^{\text{aff}}) / n$.
2. **Centering parameter:** $\sigma = (\mu_{\text{aff}} / \mu_{\text{cur}})^3$ — the cube reflects "if the affine step works well, centre less; if it doesn't, centre more". The new target is $\mu_{\text{new}} = \sigma \, \mu_{\text{cur}}$.
3. **Corrector:** re-solve with right-hand side $\sigma \mu_{\text{cur}} e - X S e - \Delta X^{\text{aff}} \Delta S^{\text{aff}} e$, picking up second-order centring terms.

The resulting algorithm typically converges in a few tens of iterations, a count that grows only slowly with problem size.

## 4. Crossover

IPM converges to an *interior* solution near the central path. To get a *basic* (vertex) solution (which simplex returns natively), run a *crossover* phase: a small simplex run starting from the IPM iterate. Many users want crossover for warm-starting subsequent solves; some don't care.

### Where IPMs live in `discopt`

- **Pure LPs (and MILPs)** go to **HiGHS** by default, and `discopt` verifies the certificate HiGHS returns (`r.algorithm_route` says `highs-lp: ...`). HiGHS decides between its simplex and IPM-plus-crossover internally. `m.solve(highs_options={...})` passes logging and search-strategy options to HiGHS (for example `{"presolve": "off"}` or `{"output_flag": True}` to print the HiGHS log), but it refuses `solver` and `run_crossover`: the route's dual certificate needs a crossover basis. So the exercises use `scipy.optimize.linprog` (`method="highs-ds"` vs `method="highs-ipm"`) when they need to choose HiGHS's algorithm. To force `discopt`'s own interior-point method instead, use `m.solve(solver="pounce")` (§5). `m.solve(milp_backend="native")` skips HiGHS and uses `discopt`'s in-house Rust simplex, with POUNCE as the fallback if the simplex declines.
- **Convex QPs and NLPs** go to **POUNCE**, a pure-Rust port of Ipopt and the default `nlp_solver`. POUNCE is a primal-dual *interior-point* method: the barrier parameter $\mu$ of §1 is driven to zero. On single-NLP routes, `r.kkt` reports the terminal residuals and the final `barrier_parameter` (see below).
- Why an exact simplex certifies LPs in `discopt`: an IPM converges *towards* the optimal face in variable space, so its objective is only accurate to about the stopping tolerance. Before the 0.9.0 fix for issue #1454, a badly scaled LP answered by the IPM reported a "certified" objective that was off by $7.5 \times 10^{-5}$. LP certificates now come from a vertex solution. This is the practical case for crossover.
- **Warm starts:** `m.solve(warm_start=previous_result)` hands POUNCE the previous point, its multipliers *and* its final $\mu$. The 0.9.0 release notes report 7 POUNCE iterations cold, 5 with the point alone, and 2 with the full primal-dual warm start, on a 1%-perturbed NLP. Restarting from a nearly-converged iterate with a tiny $\mu$ is exactly what makes IPM warm starts fragile (Exercise 5), so passing the duals and $\mu$ matters.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do, time

# Generate a tall, dense LP (a regime where IPM-style methods shine).
rng = np.random.default_rng(0)
m_, n_ = 1000, 200
A = rng.normal(size=(m_, n_))
b = A @ np.abs(rng.normal(size=n_)) + np.abs(rng.normal(size=m_))
c = rng.normal(size=n_)

mod = do.Model("tall")
# Finite upper bounds matter: discopt certifies HiGHS's answer by recomputing a
# rigorous (Neumaier-Shcherbina) dual bound from HiGHS's duals over the variable
# box. With x unbounded above that bound can be -inf, and the LP comes back as
# status "feasible" with gap_certified=False even though HiGHS solved it.
x = mod.continuous("x", shape=(n_,), lb=0, ub=10)
mod.subject_to(A @ x <= b)
mod.minimize(c @ x)

t = time.time(); r = mod.solve(); dt = time.time() - t
print(f"objective = {r.objective:.6f}   wall_time = {dt:.2f}s   status = {r.status}   certified = {r.gap_certified}")
print("route:", r.algorithm_route)          # a pure LP goes to HiGHS (verified by discopt)
print("LP iterations:", r.solver_stats.get("lp/iters"))

# A convex NLP goes to POUNCE, a primal-dual interior-point method. r.kkt shows
# the terminal residuals and the barrier parameter mu, driven (almost) to zero.
nlp = do.Model("convex_nlp")
z = nlp.continuous("z", shape=(3,), lb=0.01, ub=10)
nlp.subject_to(z[0] + z[1] + z[2] >= 3)
nlp.minimize(dm.sum(z**2) + dm.exp(z[0]) - dm.log(z[1]))
rn = nlp.solve()
print(f"NLP status = {rn.status}, objective = {rn.objective:.6f}")
print(f"final barrier parameter mu = {rn.kkt['barrier_parameter']:.2e}, "
      f"complementarity = {rn.kkt['complementarity']:.2e}")


## 5. Watching the central path: `solver="pounce"` and `r.solve_report`

`m.solve(solver="pounce")` (new in discopt 0.10) skips the routing and runs **one** POUNCE interior-point solve on the whole model. An LP, or a QP that `discopt` proves positive semidefinite, comes back `optimal`. Every other model comes back `local_optimal` with no bound, because one IPM run proves only a KKT point. The option refuses integer variables, disjunctions and callbacks: there is no tree to put them in.

On routes that make a single POUNCE call, `r.solve_report` holds POUNCE's structured report (`pounce.solve-report/v1`). Its `"iterations"` list has one row per Newton step, with the quantities from §1–§3:

| key | meaning |
|---|---|
| `objective` | $c^\top x$ at the iterate |
| `inf_pr` | primal infeasibility |
| `inf_du` | dual infeasibility (scaled) |
| `mu` | barrier parameter $\mu$ |
| `alpha_primal`, `alpha_dual` | step lengths from the fraction-to-the-boundary rule |

`r.solve_report["statistics"]` has the totals (iteration count, final KKT error, restoration calls, wall time). The report is `None` on branch-and-bound routes and on the HiGHS route, which do not make a single POUNCE call. Treat it as a diagnostic: the key names are POUNCE's, and its `inf_pr` is POUNCE's internal slack-form residual.

In [ ]:
# A small LP solved twice: HiGHS (the default route) and one POUNCE IPM solve.
rng = np.random.default_rng(0)
n, k = 20, 12
A = rng.uniform(0, 1, (k, n))
b = A @ np.full(n, 0.5) + 0.1
c = -rng.uniform(1, 2, n)

lp = do.Model("ipm_path")
x = lp.continuous("x", shape=(n,), lb=0, ub=1)
lp.subject_to(A @ x <= b, name="cap")
lp.minimize(c @ x)

r_highs = lp.solve()
r_ipm = lp.solve(solver="pounce")
print("HiGHS :", r_highs.status, f"{r_highs.objective:.12f}", "|", r_highs.algorithm_route[:22])
print("POUNCE:", r_ipm.status, f"{r_ipm.objective:.12f}", "|", r_ipm.algorithm_route)
print(f"difference = {abs(r_ipm.objective - r_highs.objective):.1e}  (the IPM stops near the vertex, not at it)")

print(f"\n{'it':>3} {'objective':>12} {'inf_pr':>9} {'inf_du':>9} {'mu':>9} {'alpha_p':>8}")
for row in r_ipm.solve_report["iterations"]:
    print(f"{row['iter']:3d} {row['objective']:12.6f} {row['inf_pr']:9.1e} {row['inf_du']:9.1e} "
          f"{row['mu']:9.1e} {row['alpha_primal']:8.2f}")
mus = [row["mu"] for row in r_ipm.solve_report["iterations"]]
print("mean mu reduction per iteration:", f"{(mus[-1] / mus[0]) ** (1 / (len(mus) - 1)):.3f}")


Read the table against §2–§3. $\mu$ falls by a roughly constant factor at each step. The primal and dual residuals fall with it, so the iterates follow the central path instead of reaching a face early. The step lengths stay just below 1 because of the fraction-to-the-boundary rule, and the solve takes about ten Newton steps. The final objective agrees with HiGHS's vertex to about $10^{-9}$, not to machine precision. That is why `discopt` certifies LPs from a vertex (§4), and why `solver="pounce"` is a tool for studying the method rather than the default.

## 6. When IPM beats simplex (and vice versa)

| Structure | Winner |
|-----------|--------|
| Sparse, well-conditioned, "few" constraints active | simplex |
| Dense, many constraints, high dim | IPM |
| Re-solving with small perturbation | simplex (warm start) |
| First solve, large LP | IPM |
| Need basic solution | simplex (or IPM + crossover) |

This trade-off is the practical flip side of the polynomial-vs-exponential theory.

## References
{cite:p}`Karmarkar1984` (the breakthrough), {cite:p}`Mehrotra1992` (practical), {cite:p}`Wright1997` (textbook), {cite:p}`Khachiyan1979` (the earlier ellipsoid result).